In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install xgboost shap plotly --quiet

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pd.set_option('display.max_columns',None)

In [ ]:
df = pd.read_csv(
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['Churn'].value_counts()

In [ ]:
df['Churn'].value_counts(normalize=True)*100

In [ ]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
df.isnull().sum()

In [ ]:
df[df['TotalCharges'].isnull()]

In [ ]:
df = df.dropna()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop(
    'customerID',
    axis=1,
    inplace=True
)

In [ ]:
df.head()

In [ ]:
sns.countplot(
    x='Churn',
    data=df
)

plt.show()

In [ ]:
churn_rate = (
    df['Churn']
    .value_counts(normalize=True)
    *100
)

print(churn_rate)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x='Contract',
    hue='Churn',
    data=df
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    x='Churn',
    y='MonthlyCharges',
    data=df
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    data=df,
    x='tenure',
    hue='Churn',
    multiple='stack'
)

plt.show()

In [ ]:
numeric_df = df.select_dtypes(
    include=np.number
)

In [ ]:
print(numeric_df.columns)

In [ ]:
plt.figure(figsize=(10,7))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.show()

In [ ]:
service_cols = [
'PhoneService',
'MultipleLines',
'OnlineSecurity',
'OnlineBackup',
'DeviceProtection',
'TechSupport',
'StreamingTV',
'StreamingMovies'
]

df['ServiceCount'] = (
    df[service_cols]
    .apply(
        lambda row:
        sum(row=='Yes'),
        axis=1
    )
)

In [ ]:
df['ChargesPerMonth'] = (
    df['TotalCharges']
    /(df['tenure']+1)
)

In [ ]:
df.isnull().sum()

In [ ]:
df['Churn'] = df['Churn'].map({
    'Yes':1,
    'No':0
})

In [ ]:
df['Churn'].value_counts()

In [ ]:
X = df.drop('Churn',axis=1)

y = df['Churn']

In [ ]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [ ]:
X.shape


In [ ]:
X.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression(
    max_iter=2000
)

lr.fit(
    X_train,
    y_train
)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [ ]:
lr = LogisticRegression(
    max_iter=5000,
    random_state=42
)

lr.fit(
    X_train_scaled,
    y_train
)

In [ ]:
y_pred_lr = lr.predict(
    X_test_scaled
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

def evaluate_model(
        y_true,
        y_pred,
        model_name):

    print(f"\n===== {model_name} =====")

    print(
        "Accuracy:",
        accuracy_score(
            y_true,
            y_pred
        )
    )

    print(
        "Precision:",
        precision_score(
            y_true,
            y_pred
        )
    )

    print(
        "Recall:",
        recall_score(
            y_true,
            y_pred
        )
    )

    print(
        "F1:",
        f1_score(
            y_true,
            y_pred
        )
    )

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            y_pred
        )
    )

In [ ]:
evaluate_model(
    y_test,
    y_pred_lr,
    "Scaled Logistic Regression"
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
evaluate_model(
    y_test,
    y_pred_rf,
    "Random Forest"
)

In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_xgb = xgb.predict(
    X_test
)

In [ ]:
evaluate_model(
    y_test,
    y_pred_xgb,
    "XGBoost"
)

In [ ]:
results = pd.DataFrame({

'Model':[
'Logistic Regression',
'Random Forest',
'XGBoost'
],

'Accuracy':[

accuracy_score(
y_test,
y_pred_lr
),

accuracy_score(
y_test,
y_pred_rf
),

accuracy_score(
y_test,
y_pred_xgb
)
],

'Recall':[

recall_score(
y_test,
y_pred_lr
),

recall_score(
y_test,
y_pred_rf
),

recall_score(
y_test,
y_pred_xgb
)
],

'F1':[

f1_score(
y_test,
y_pred_lr
),

f1_score(
y_test,
y_pred_rf
),

f1_score(
y_test,
y_pred_xgb
)
]

})

results

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
lr_cv_scores = cross_val_score(
    lr,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='f1'
)

print("CV Scores:", lr_cv_scores)
print("Mean CV Score:", lr_cv_scores.mean())

In [ ]:
rf_cv_scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("CV Scores:", rf_cv_scores)
print("Mean CV Score:", rf_cv_scores.mean())

In [ ]:
xgb_cv_scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("CV Scores:", xgb_cv_scores)
print("Mean CV Score:", xgb_cv_scores.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {

'n_estimators':[100,200],

'max_depth':[5,10,None],

'min_samples_split':[2,5]

}

In [ ]:
grid_rf = GridSearchCV(

RandomForestClassifier(
random_state=42
),

param_grid,

cv=5,

scoring='f1',

n_jobs=-1

)

In [ ]:
grid_rf.fit(
    X_train,
    y_train
)

In [ ]:
print(
    grid_rf.best_params_
)

In [ ]:
print(
    grid_rf.best_score_
)

In [ ]:
best_rf = grid_rf.best_estimator_

In [ ]:
y_pred_best_rf = best_rf.predict(
    X_test
)

evaluate_model(
    y_test,
    y_pred_best_rf,
    "Tuned Random Forest"
)

In [ ]:
feature_importance = pd.DataFrame({

'Feature':X.columns,

'Importance':
best_rf.feature_importances_

})

In [ ]:
feature_importance = (
feature_importance
.sort_values(
'Importance',
ascending=False
)
)

In [ ]:
feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10,7))

sns.barplot(

x='Importance',

y='Feature',

data=feature_importance.head(15)

)

plt.show()

In [ ]:
!pip install shap -q

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(
    best_rf
)

In [ ]:
shap_values = explainer.shap_values(
    X_test
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test
)

In [ ]:
shap_explanation_churn = shap.Explanation(
    values=shap_values[:, :, 1],
    base_values=explainer.expected_value[1],
    data=X_test
)

shap.plots.beeswarm(shap_explanation_churn)

In [ ]:
customer_index = 0


single_customer_explanation = shap.Explanation(
    values=shap_values[customer_index, :, 1],
    base_values=explainer.expected_value[1],
    data=X_test.iloc[customer_index]
)

shap.plots.waterfall(single_customer_explanation)

In [ ]:
shap_explanation = explainer(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)

results_summary = pd.DataFrame({

'Model':[

'Scaled Logistic Regression',
'Random Forest',
'XGBoost',
'Tuned Random Forest'

],

'Accuracy':[

accuracy_score(
    y_test,
    y_pred_lr
),

accuracy_score(
    y_test,
    y_pred_rf
),

accuracy_score(
    y_test,
    y_pred_xgb
),

accuracy_score(
    y_test,
    y_pred_best_rf
)

],

'Precision':[

precision_score(
    y_test,
    y_pred_lr
),

precision_score(
    y_test,
    y_pred_rf
),

precision_score(
    y_test,
    y_pred_xgb
),

precision_score(
    y_test,
    y_pred_best_rf
)

],

'Recall':[

recall_score(
    y_test,
    y_pred_lr
),

recall_score(
    y_test,
    y_pred_rf
),

recall_score(
    y_test,
    y_pred_xgb
),

recall_score(
    y_test,
    y_pred_best_rf
)

],

'F1':[

f1_score(
    y_test,
    y_pred_lr
),

f1_score(
    y_test,
    y_pred_rf
),

f1_score(
    y_test,
    y_pred_xgb
),

f1_score(
    y_test,
    y_pred_best_rf
)

]

})

results_summary

In [ ]:
y_prob_lr = lr.predict_proba(
    X_test_scaled
)[:,1]

In [ ]:
y_prob_rf = rf.predict_proba(
    X_test
)[:,1]

In [ ]:
y_prob_xgb = xgb.predict_proba(
    X_test
)[:,1]

In [ ]:
y_prob_best_rf = best_rf.predict_proba(
    X_test
)[:,1]

In [ ]:
results_summary['ROC_AUC'] = [

roc_auc_score(
    y_test,
    y_prob_lr
),

roc_auc_score(
    y_test,
    y_prob_rf
),

roc_auc_score(
    y_test,
    y_prob_xgb
),

roc_auc_score(
    y_test,
    y_prob_best_rf
)

]

results_summary.round(4)

In [ ]:
lr_balanced = LogisticRegression(

max_iter=5000,

class_weight='balanced',

random_state=42

)

lr_balanced.fit(
    X_train_scaled,
    y_train
)

y_pred_balanced = lr_balanced.predict(
    X_test_scaled
)

In [ ]:
evaluate_model(
    y_test,
    y_pred_balanced,
    "Balanced Logistic Regression"
)

In [ ]:
y_prob = lr.predict_proba(
    X_test_scaled
)[:,1]

threshold = 0.35

y_pred_custom = (
    y_prob >= threshold
).astype(int)

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
lr_balanced = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    random_state=42
)

lr_balanced.fit(
    X_train_scaled,
    y_train
)

y_pred_balanced = lr_balanced.predict(
    X_test_scaled
)

evaluate_model(
    y_test,
    y_pred_balanced,
    "Balanced Logistic Regression"
)